## 0. Validación de datos

In [ ]:
from pathlib import Path
import pandas as pd

VISION_DIR = Path("/kaggle/working/data/vision")

tracks_df = pd.read_csv(VISION_DIR / "pred_tracks_all_videos.csv")

print("Shape:", tracks_df.shape)
print("Columnas:")
print(tracks_df.columns.tolist())

print("\nParticipantes con tracks:", tracks_df["id"].nunique())
print("Videos con tracks:", tracks_df["video_filename"].nunique())
print("Tracks únicos:", tracks_df["track_uid"].nunique())

display(tracks_df.head())

display(
    tracks_df
    .groupby(["id", "video_filename"])
    .agg(
        n_points=("frame", "count"),
        n_frames=("frame", "nunique"),
        n_tracks=("track_uid", "nunique"),
        mean_confidence=("confidence", "mean"),
    )
    .reset_index()
    .sort_values("n_tracks", ascending=False)
    #.head(20)
)

Shape: (12792993, 12)
Columnas:
['id', 'video', 'video_filename', 'frame', 'track_id', 'x_center', 'y_center', 'width', 'height', 'confidence', 'source_video_path', 'track_uid']

Participantes con tracks: 85
Videos con tracks: 85
Tracks únicos: 391423


,id,video,video_filename,frame,track_id,x_center,y_center,width,height,confidence,source_video_path,track_uid
0,72,72.0,72_11.02.24_JMA.avi,0.0,1.0,0.174512,0.895312,0.026172,0.029167,0.778809,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__1
1,72,72.0,72_11.02.24_JMA.avi,0.0,2.0,0.550000,0.210156,0.037500,0.038802,0.771973,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__2
2,72,72.0,72_11.02.24_JMA.avi,0.0,3.0,0.462500,0.391927,0.029687,0.033333,0.763672,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__3
3,72,72.0,72_11.02.24_JMA.avi,0.0,4.0,0.349219,0.660937,0.030859,0.032292,0.759277,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__4
4,72,72.0,72_11.02.24_JMA.avi,0.0,5.0,0.675000,0.105599,0.028125,0.032031,0.756348,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__5


,id,video_filename,n_points,n_frames,n_tracks,mean_confidence
30,31,31_10.04.22_JMA.avi,166033,3604,19181,0.585978
26,27,27_09.02.18_IVS.avi,167460,3595,17815,0.714132
82,83,83_11.11.09_HH.avi,168356,3579,16713,0.652356
25,26,26_09.02.12_IVS.avi,168429,3435,16503,0.676813
15,16,16_09.01.28_SSW.avi,166123,3444,16167,0.680954
...,...,...,...,...,...,...
51,52,52_09.12.03_drift_SSW.avi,75676,5762,230,0.657694
46,47,47_09.10.19_SSW.avi,37257,5969,214,0.602239
13,14,14_09.01.27_SSW.avi,26502,5989,111,0.616224
22,23,23_09.02.04_SSW.avi,26250,5683,91,0.620507


## 1. Setup

In [20]:
import numpy as np
import re

RAW_DIR = Path("/kaggle/working/data/raw")
VISION_DIR = Path("/kaggle/working/data/vision")
PROCESSED_DIR = Path("/kaggle/working/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FPS = 50
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

def clean_col_name(col: str) -> str:
    col = col.strip()
    col = col.replace("µ", "u")
    col = col.replace("²", "2")
    col = col.replace("⁶", "6")
    col = col.replace(":", "_")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace(",", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("%", "pct")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[^a-zA-Z0-9_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col.lower().strip("_")

tracks_path = VISION_DIR / "pred_tracks_all_videos.csv"
tracks_df = pd.read_csv(tracks_path)

tracks_df.head()

,id,video,video_filename,frame,track_id,x_center,y_center,width,height,confidence,source_video_path,track_uid
0,72,72.0,72_11.02.24_JMA.avi,0.0,1.0,0.174512,0.895312,0.026172,0.029167,0.778809,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__1
1,72,72.0,72_11.02.24_JMA.avi,0.0,2.0,0.550000,0.210156,0.037500,0.038802,0.771973,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__2
2,72,72.0,72_11.02.24_JMA.avi,0.0,3.0,0.462500,0.391927,0.029687,0.033333,0.763672,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__3
3,72,72.0,72_11.02.24_JMA.avi,0.0,4.0,0.349219,0.660937,0.030859,0.032292,0.759277,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__4
4,72,72.0,72_11.02.24_JMA.avi,0.0,5.0,0.675000,0.105599,0.028125,0.032031,0.756348,/kaggle/input/datasets/stevenhicks/visem-video...,72_11.02.24_JMA.avi__5


## 2. Normaliza columnas

In [21]:
tracks_df["video"] = tracks_df["video"].astype(str)
tracks_df["track_id"] = tracks_df["track_id"].astype(str)

if "track_uid" not in tracks_df.columns:
    tracks_df["track_uid"] = (
        tracks_df["video"].astype(str)
        + "__"
        + tracks_df["track_id"].astype(str)
    )

tracks_df = tracks_df.sort_values(
    ["video", "track_uid", "frame"]
).reset_index(drop=True)

## 3. Calcula movimiento por punto:

In [22]:
def add_motion_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["x_px"] = df["x_center"] * FRAME_WIDTH
    df["y_px"] = df["y_center"] * FRAME_HEIGHT

    df["prev_x_px"] = df.groupby("track_uid")["x_px"].shift(1)
    df["prev_y_px"] = df.groupby("track_uid")["y_px"].shift(1)
    df["prev_frame"] = df.groupby("track_uid")["frame"].shift(1)

    df["dt_frames"] = df["frame"] - df["prev_frame"]
    df["dt_seconds"] = df["dt_frames"] / FPS

    df["dx_px"] = df["x_px"] - df["prev_x_px"]
    df["dy_px"] = df["y_px"] - df["prev_y_px"]

    df["step_distance_px"] = np.sqrt(
        df["dx_px"] ** 2 + df["dy_px"] ** 2
    )

    df["speed_px_s"] = df["step_distance_px"] / df["dt_seconds"]

    # Evitar saltos raros o divisiones inválidas
    df.loc[df["dt_frames"] <= 0, "speed_px_s"] = np.nan
    df.loc[~np.isfinite(df["speed_px_s"]), "speed_px_s"] = np.nan

    return df


tracks_motion = add_motion_columns(tracks_df)
tracks_motion.head()

,id,video,video_filename,frame,track_id,x_center,y_center,width,height,confidence,...,y_px,prev_x_px,prev_y_px,prev_frame,dt_frames,dt_seconds,dx_px,dy_px,step_distance_px,speed_px_s
0,1,1.0,1_09.09.02_SSW.avi,0.0,1.0,0.525781,0.021208,0.032812,0.041569,0.769043,...,10.179688,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1.0,1_09.09.02_SSW.avi,1.0,1.0,0.525103,0.021024,0.032486,0.041145,0.771973,...,10.091555,336.500015,10.179688,0.0,1.0,0.02,-0.433922,-0.088133,0.442782,22.139080
2,1,1.0,1_09.09.02_SSW.avi,2.0,1.0,0.524987,0.021005,0.032536,0.041200,0.756348,...,10.082602,336.066093,10.091555,1.0,1.0,0.02,-0.074158,-0.008953,0.074696,3.734811
3,1,1.0,1_09.09.02_SSW.avi,3.0,1.0,0.525262,0.020916,0.032434,0.041051,0.777344,...,10.039597,335.991936,10.082602,2.0,1.0,0.02,0.175629,-0.043005,0.180817,9.040857
4,1,1.0,1_09.09.02_SSW.avi,4.0,1.0,0.525358,0.021058,0.032523,0.041101,0.778809,...,10.107709,336.167564,10.039597,3.0,1.0,0.02,0.061569,0.068112,0.091815,4.590756


## 4. Resume
Por trayectoria

In [23]:
def summarize_tracks(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for track_uid, g in df.groupby("track_uid"):
        g = g.sort_values("frame")

        video = g["video"].iloc[0]
        n_frames = g["frame"].nunique()
        first_frame = g["frame"].min()
        last_frame = g["frame"].max()
        duration_frames = last_frame - first_frame + 1
        duration_seconds = duration_frames / FPS

        path_length_px = g["step_distance_px"].sum(skipna=True)

        start_x = g["x_px"].iloc[0]
        start_y = g["y_px"].iloc[0]
        end_x = g["x_px"].iloc[-1]
        end_y = g["y_px"].iloc[-1]

        net_displacement_px = np.sqrt(
            (end_x - start_x) ** 2 + (end_y - start_y) ** 2
        )

        straightness = (
            net_displacement_px / path_length_px
            if path_length_px > 0
            else np.nan
        )

        rows.append({
            "video": video,
            "track_uid": track_uid,
            "track_n_frames": n_frames,
            "track_duration_seconds": duration_seconds,
            "track_path_length_px": path_length_px,
            "track_net_displacement_px": net_displacement_px,
            "track_straightness": straightness,
            "track_mean_speed_px_s": g["speed_px_s"].mean(),
            "track_median_speed_px_s": g["speed_px_s"].median(),
            "track_max_speed_px_s": g["speed_px_s"].max(),
            "track_speed_std_px_s": g["speed_px_s"].std(),
            "track_mean_confidence": g["confidence"].mean() if "confidence" in g.columns else np.nan,
            "track_mean_area_norm": (g["width"] * g["height"]).mean(),
        })

    return pd.DataFrame(rows)


track_summary = summarize_tracks(tracks_motion)
track_summary.head()

,video,track_uid,track_n_frames,track_duration_seconds,track_path_length_px,track_net_displacement_px,track_straightness,track_mean_speed_px_s,track_median_speed_px_s,track_max_speed_px_s,track_speed_std_px_s,track_mean_confidence,track_mean_area_norm
0,10.0,10_12.03.12_minor drift_HH.avi__1,17,0.42,30.505731,13.740472,0.450423,75.012439,69.738440,126.828554,35.420150,0.444795,0.002024
1,10.0,10_12.03.12_minor drift_HH.avi__10,95,2.36,75.960327,9.186645,0.120940,35.336957,23.183350,182.436802,34.572311,0.382090,0.002622
2,10.0,10_12.03.12_minor drift_HH.avi__1013,24,1.30,29.044622,5.806997,0.199934,37.467742,28.274981,122.995821,29.763377,0.457194,0.002114
3,10.0,10_12.03.12_minor drift_HH.avi__1018,12,0.26,18.968146,14.052814,0.740864,78.364116,78.169372,105.664955,17.296040,0.454386,0.001490
4,10.0,10_12.03.12_minor drift_HH.avi__1020,3,0.24,3.727968,1.231555,0.330356,32.994280,32.994280,58.214089,35.666196,0.350830,0.000603


Por video

In [25]:
def q(series, value):
    return series.quantile(value)


video_features = (
    track_summary
    .groupby("video")
    .agg(
        video_n_tracks=("track_uid", "nunique"),

        video_mean_track_frames=("track_n_frames", "mean"),
        video_median_track_frames=("track_n_frames", "median"),
        video_p90_track_frames=("track_n_frames", lambda x: q(x, 0.90)),

        video_mean_duration_s=("track_duration_seconds", "mean"),
        video_median_duration_s=("track_duration_seconds", "median"),

        video_mean_speed_px_s=("track_mean_speed_px_s", "mean"),
        video_median_speed_px_s=("track_median_speed_px_s", "median"),
        video_p90_speed_px_s=("track_mean_speed_px_s", lambda x: q(x, 0.90)),
        video_speed_std_px_s=("track_mean_speed_px_s", "std"),

        video_mean_displacement_px=("track_net_displacement_px", "mean"),
        video_median_displacement_px=("track_net_displacement_px", "median"),

        video_mean_path_length_px=("track_path_length_px", "mean"),
        video_mean_straightness=("track_straightness", "mean"),
        video_median_straightness=("track_straightness", "median"),

        video_mean_confidence=("track_mean_confidence", "mean"),
        video_mean_area_norm=("track_mean_area_norm", "mean"),
    )
    .reset_index()
)

## 5. Agrega proporciones útiles

In [27]:
LONG_TRACK_MIN_FRAMES = 50
GLOBAL_LOW_SPEED_THRESHOLD = track_summary["track_mean_speed_px_s"].quantile(0.25)
GLOBAL_FAST_SPEED_THRESHOLD = track_summary["track_mean_speed_px_s"].quantile(0.75)

tmp = track_summary.assign(
    is_long_track=track_summary["track_n_frames"] >= LONG_TRACK_MIN_FRAMES,
    is_fast_track=track_summary["track_mean_speed_px_s"] >= GLOBAL_FAST_SPEED_THRESHOLD,
    is_low_movement_track=track_summary["track_mean_speed_px_s"] <= GLOBAL_LOW_SPEED_THRESHOLD,
    is_progressive_like=(
        (track_summary["track_straightness"] >= 0.5)
        & (track_summary["track_mean_speed_px_s"] >= GLOBAL_FAST_SPEED_THRESHOLD)
    )
)

extra = (
    tmp
    .groupby("video", as_index=False)
    .agg(
        video_long_track_ratio=("is_long_track", "mean"),
        video_fast_track_ratio=("is_fast_track", "mean"),
        video_low_movement_track_ratio=("is_low_movement_track", "mean"),
        video_progressive_like_ratio=("is_progressive_like", "mean"),
    )
)

video_features = video_features.merge(extra, on="video", how="left")
video_features.head()

,video,video_n_tracks,video_mean_track_frames,video_median_track_frames,video_p90_track_frames,video_mean_duration_s,video_median_duration_s,video_mean_speed_px_s,video_median_speed_px_s,video_p90_speed_px_s,...,video_mean_confidence,video_mean_area_norm,video_long_track_ratio_x,video_fast_track_ratio_x,video_low_movement_track_ratio_x,video_progressive_like_ratio_x,video_long_track_ratio_y,video_fast_track_ratio_y,video_low_movement_track_ratio_y,video_progressive_like_ratio_y
0,1.0,1669,103.094667,23.0,273.0,2.507501,0.62,72.072618,69.112641,125.790869,...,0.611363,0.001079,0.335530,0.173757,0.267825,0.121630,0.335530,0.173757,0.267825,0.121630
1,10.0,2449,33.994283,8.0,74.0,1.096407,0.26,47.963816,40.440661,84.184319,...,0.438921,0.001548,0.137607,0.030625,0.445080,0.028991,0.137607,0.030625,0.445080,0.028991
2,11.0,548,308.821168,55.0,1187.5,6.786241,1.86,56.406181,20.780447,149.073418,...,0.564451,0.001139,0.512774,0.171533,0.540146,0.138686,0.512774,0.171533,0.540146,0.138686
3,12.0,2702,54.430052,13.0,110.0,1.231584,0.32,152.239625,156.694921,213.811101,...,0.573948,0.001793,0.216506,0.762768,0.047742,0.521466,0.216506,0.762768,0.047742,0.521466
4,13.0,460,369.169565,138.0,1472.8,7.728696,3.04,54.005838,38.210061,110.194941,...,0.605753,0.001422,0.663043,0.100000,0.471739,0.067391,0.663043,0.100000,0.471739,0.067391


Guarda:

In [28]:
video_features.to_csv(
    VISION_DIR / "motion_features_by_video.csv",
    index=False
)

track_summary.to_csv(
    VISION_DIR / "motion_features_by_track.csv",
    index=False
)

En videos.csv, se tiene algo así:

ID;video
1;1_09.09.02_SSW.avi

En el notebook de tracking, el campo video puede venir como:
1
o como otro identificador interno.

Entonces se debe crear una llave compatible.

In [30]:
def make_video_key(s):
    """
    Convierte valores como:
    1
    1.0
    "1"
    "1_09.09.02_SSW.avi"

    en una llave entera común:
    1
    """
    return (
        s.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.extract(r"^(\d+)", expand=False)
        .astype("Int64")
    )

In [35]:
videos_df = pd.read_csv(
    "/kaggle/input/datasets/stevenhicks/visem-video-dataset/visem-dataset/videos.csv",
    sep=";",
    decimal=","
)

videos_df.columns = [clean_col_name(c) for c in videos_df.columns]

videos_df = videos_df.rename(columns={
    "video": "video_filename"
})

videos_df["id"] = videos_df["id"].astype("Int64")
videos_df["video_key"] = make_video_key(videos_df["video_filename"])

video_map = (
    videos_df[["id", "video_filename", "video_key"]]
    .drop_duplicates("video_key")
)

video_features = pd.read_csv(VISION_DIR / "motion_features_by_video.csv")

video_features["video_key"] = make_video_key(video_features["video"])

video_features_with_id = (
    video_features
    .merge(
        video_map,
        on="video_key",
        how="left",
        validate="m:1"
    )
)

video_features_with_id.head()

,video,video_n_tracks,video_mean_track_frames,video_median_track_frames,video_p90_track_frames,video_mean_duration_s,video_median_duration_s,video_mean_speed_px_s,video_median_speed_px_s,video_p90_speed_px_s,...,video_fast_track_ratio_x,video_low_movement_track_ratio_x,video_progressive_like_ratio_x,video_long_track_ratio_y,video_fast_track_ratio_y,video_low_movement_track_ratio_y,video_progressive_like_ratio_y,video_key,id,video_filename
0,1.0,1669,103.094667,23.0,273.0,2.507501,0.62,72.072618,69.112641,125.790869,...,0.173757,0.267825,0.121630,0.335530,0.173757,0.267825,0.121630,1,1,1_09.09.02_SSW.avi
1,10.0,2449,33.994283,8.0,74.0,1.096407,0.26,47.963816,40.440661,84.184319,...,0.030625,0.445080,0.028991,0.137607,0.030625,0.445080,0.028991,10,10,10_12.03.12_minor drift_HH.avi
2,11.0,548,308.821168,55.0,1187.5,6.786241,1.86,56.406181,20.780447,149.073418,...,0.171533,0.540146,0.138686,0.512774,0.171533,0.540146,0.138686,11,11,11_09.01.23_JMA.avi
3,12.0,2702,54.430052,13.0,110.0,1.231584,0.32,152.239625,156.694921,213.811101,...,0.762768,0.047742,0.521466,0.216506,0.762768,0.047742,0.521466,12,12,12_09.01.23_SSW.avi
4,13.0,460,369.169565,138.0,1472.8,7.728696,3.04,54.005838,38.210061,110.194941,...,0.100000,0.471739,0.067391,0.663043,0.100000,0.471739,0.067391,13,13,13_09.01.26_SSW.avi


Valida y si hay muchos NaN, hay que revisar cómo el notebook está nombrando los videos:

In [36]:
print("Videos con ID encontrado:")
print(video_features_with_id["id"].notna().sum())

print("Videos sin ID encontrado:")
display(video_features_with_id[video_features_with_id["id"].isna()].head())

Videos con ID encontrado:
85
Videos sin ID encontrado:


,video,video_n_tracks,video_mean_track_frames,video_median_track_frames,video_p90_track_frames,video_mean_duration_s,video_median_duration_s,video_mean_speed_px_s,video_median_speed_px_s,video_p90_speed_px_s,...,video_fast_track_ratio_x,video_low_movement_track_ratio_x,video_progressive_like_ratio_x,video_long_track_ratio_y,video_fast_track_ratio_y,video_low_movement_track_ratio_y,video_progressive_like_ratio_y,video_key,id,video_filename


## 6. Si hay más de un video por participante 

In [37]:
numeric_video_cols = video_features_with_id.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_video_cols = [
    c for c in numeric_video_cols
    if c not in ["id"]
]

video_features_by_participant = (
    video_features_with_id
    .groupby("id")[numeric_video_cols]
    .mean()
    .reset_index()
)

video_features_by_participant.to_csv(
    VISION_DIR / "motion_features_by_participant.csv",
    index=False
)

video_features_by_participant.head()

,id,video,video_n_tracks,video_mean_track_frames,video_median_track_frames,video_p90_track_frames,video_mean_duration_s,video_median_duration_s,video_mean_speed_px_s,video_median_speed_px_s,...,video_mean_area_norm,video_long_track_ratio_x,video_fast_track_ratio_x,video_low_movement_track_ratio_x,video_progressive_like_ratio_x,video_long_track_ratio_y,video_fast_track_ratio_y,video_low_movement_track_ratio_y,video_progressive_like_ratio_y,video_key
0,1,1.0,1669.0,103.094667,23.0,273.0,2.507501,0.62,72.072618,69.112641,...,0.001079,0.335530,0.173757,0.267825,0.121630,0.335530,0.173757,0.267825,0.121630,1.0
1,2,2.0,487.0,292.383984,45.0,1124.0,6.402834,1.38,53.087243,36.630727,...,0.001394,0.490760,0.129363,0.476386,0.088296,0.490760,0.129363,0.476386,0.088296,2.0
2,3,3.0,958.0,179.463466,33.0,683.9,3.948559,0.91,60.896325,56.376456,...,0.001153,0.448852,0.125261,0.378914,0.104384,0.448852,0.125261,0.378914,0.104384,3.0
3,4,4.0,1110.0,157.421622,32.5,523.2,3.734144,1.03,59.091332,39.203050,...,0.001130,0.439640,0.177477,0.470270,0.145946,0.439640,0.177477,0.470270,0.145946,4.0
4,5,5.0,4105.0,41.827284,7.0,65.0,1.107016,0.20,138.252964,138.180108,...,0.001416,0.121559,0.635810,0.108892,0.535932,0.121559,0.635810,0.108892,0.535932,5.0


## 7. Crear dataset multimodal final

In [3]:
motion_features = pd.read_csv(
    VISION_DIR / "motion_features_by_participant.csv"
)

targets = pd.read_csv(
    PROCESSED_DIR / "semen_targets.csv"
)

clinical_features = pd.read_csv(
    PROCESSED_DIR / "clinical_features.csv"
)

multimodal_dataset = (
    clinical_features
    .merge(motion_features, on="id", how="left")
    .merge(targets, on="id", how="inner")
)

multimodal_dataset.to_csv(
    PROCESSED_DIR / "multimodal_dataset.csv",
    index=False
)

multimodal_dataset.head()

,id,participant_abstinence_timedays,participant_body_mass_index_kg_m2,participant_age_years,serum_serum_c14_0_myristic_acid,serum_serum_c16_0_palmitic_acid,serum_serum_c16_1_palmitoleic_acid,serum_serum_c18_0_stearic_acid,serum_serum_c18_1_n_9_oleic_acid,serum_serum_total_c18_1,...,video_low_movement_track_ratio_y,video_progressive_like_ratio_y,video_key,target_low_concentration,target_low_total_sperm_count,target_low_progressive_motility,target_low_total_motility,target_low_vitality,target_low_morphology,target_any_semen_abnormality
0,1,"4,0",32.5,36,0.36,29.72,0.64,13.67,9.00,10.90,...,0.267825,0.121630,1.0,0,0,0,0,0,1,1
1,2,"4,0",33.7,61,0.28,31.22,0.47,11.84,9.02,11.29,...,0.476386,0.088296,2.0,0,0,1,1,1,1,1
2,3,"2,0",62.7,51,0.36,27.95,0.47,16.57,8.65,10.36,...,0.378914,0.104384,3.0,0,0,1,0,0,0,1
3,4,"2,5",45.5,38,0.32,28.10,0.64,16.09,8.56,10.13,...,0.470270,0.145946,4.0,0,0,0,0,0,1,1
4,5,"3,0",51.0,33,0.40,29.94,0.80,14.17,9.47,11.02,...,0.108892,0.535932,5.0,0,0,0,0,0,1,1


## 8. Crear X y Y

In [4]:
target_name = "target_any_semen_abnormality"

drop_cols = [
    "id",
    "video_filename",
]

target_cols = [c for c in multimodal_dataset.columns if c.startswith("target_")]

X = multimodal_dataset.drop(columns=drop_cols + target_cols)
y = multimodal_dataset[target_name]

X.to_csv(PROCESSED_DIR / "X_multimodal.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_target_any_semen_abnormality.csv", index=False)

print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts())

X: (85, 76)
y: (85,)
target_any_semen_abnormality
1    59
0    26
Name: count, dtype: int64


## 9. Validaciones obligatorias
Si id tiene duplicados, todavía no tienes una tabla limpia de una fila por participante.

In [5]:
print("Filas totales:", len(multimodal_dataset))
print("Participantes únicos:", multimodal_dataset["id"].nunique())

print("\nDuplicados por ID:")
display(
    multimodal_dataset["id"]
    .value_counts()
    .loc[lambda s: s > 1]
)

print("\nValores faltantes por columna:")
missing_report = (
    multimodal_dataset
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_ratio")
)

display(missing_report.head(30))

print("\nDistribución del target:")
display(multimodal_dataset[target_name].value_counts(dropna=False))
display(multimodal_dataset[target_name].value_counts(normalize=True, dropna=False))

Filas totales: 85
Participantes únicos: 85

Duplicados por ID:


Series([], Name: count, dtype: int64)


Valores faltantes por columna:


,missing_ratio
id,0.0
participant_abstinence_timedays,0.0
participant_body_mass_index_kg_m2,0.0
participant_age_years,0.0
serum_serum_c14_0_myristic_acid,0.0
serum_serum_c16_0_palmitic_acid,0.0
serum_serum_c16_1_palmitoleic_acid,0.0
serum_serum_c18_0_stearic_acid,0.0
serum_serum_c18_1_n_9_oleic_acid,0.0
serum_serum_total_c18_1,0.0



Distribución del target:


target_any_semen_abnormality
1    59
0    26
Name: count, dtype: int64

target_any_semen_abnormality
1    0.694118
0    0.305882
Name: proportion, dtype: float64